# 02 Ă‚Â· Codex chooses taught manipulation skills

Your signed-in Codex CLI sees the two camera images and chooses from a small catalog:
`pick_mouse`, `place_left`, `place_right`, and `place_middle`. When the scene or
teaching is insufficient, it returns a non-executable explanation.
Local code turns the selected skills into a bounded trajectory of taught poses.

**Kernel:** `Ramion Ă‚Â· setup / Codex`  
**Credentials:** use your own Linux account and dedicated Codex profile.

If Codex is not installed in your account, install it from the [official CLI instructions](https://developers.openai.com/codex/cli/) first. Then create a dedicated profile from your own Linux account in an SSH terminal:

```bash
install -d -m 700 "$HOME/.local/state/ramion/codex"
CODEX_HOME="$HOME/.local/state/ramion/codex" codex -c 'cli_auth_credentials_store="file"' login --device-auth
chmod 600 "$HOME/.local/state/ramion/codex/auth.json"
```

Enable device-code sign-in in your ChatGPT security settings if required, then sign in with your own Codex account. Use
the profile path `~/.local/state/ramion/codex` in the notebook's `CODEX_PROFILE`.

Teach the poses during a supervised physical session. Codex does not invent joint angles.
The taught mouse pickup position and target regions must still match the live scene.


In [ ]:
import os
from pathlib import Path
from soarm_lab.notebooks.client import WorkbenchClient

REPO = next((p for base in (Path.cwd(), *Path.cwd().parents)
             for p in (base, base / 'apps' / 'arm-workbench')
             if (p / 'packages' / 'soarm').is_dir()), None)
if REPO is None:
    raise RuntimeError('Open this notebook from the Ramion checkout.')
WORKSPACE = REPO / 'workspace' / 'notebooks'
WORKSPACE.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('HF_HOME', str(REPO / 'workspace' / 'huggingface'))
client = WorkbenchClient(os.environ.get('RAMION_WORKBENCH_URL', 'http://100.85.56.14:8765'))
TASK = 'Move the stuffed toy mouse onto the left gray area.'


In [ ]:
from soarm_lab.notebooks.codex import CodexPlanner, PoseLibrary

library = PoseLibrary(WORKSPACE / 'mouse_poses.json')
CODEX_PROFILE = ''  # Set your dedicated profile directory after device login.
planner = CodexPlanner(profile=CODEX_PROFILE or None)


## 1. Teach the skill library using the workbench

Use the workbench's existing manual controls to reach each pose, then capture its
**measured** joints below. Capturing a pose only saves feedback; it sends no motion.
Use collision-clear, slow paths and keep the toy at the same initial pickup position.

| Pose | Meaning |
| --- | --- |
| `mouse_approach` | Open gripper safely above the toy |
| `mouse_grasp_open` | Gripper around the toy, still open |
| `mouse_grasp_closed` | Toy grasped, without excessive compression |
| `mouse_lift` | Toy lifted clear of the table |
| `left_approach`, `right_approach`, `middle_approach` | Grasped toy above the selected destination |
| `left_place_closed`, `right_place_closed`, `middle_place_closed` | Toy lowered onto the destination, gripper still closed |
| `left_release`, `right_release`, `middle_release` | Same arm pose as place_closed; only the gripper opens |
| `left_retreat`, `right_retreat`, `middle_retreat` | Open gripper clear of the placed toy |

Left/right refer to the overhead image. Middle means between the two gray areas.
The complete catalog has 16 poses. You can teach `pick_mouse` and one destination
first; the planner only receives skills whose poses are complete. Keep the arm joints unchanged when closing at the mouse
or opening at a destination, and keep the gripper unchanged during transport. The
library checks those transitions. Check the whole path too; endpoints alone do not
prove collision clearance.


In [ ]:
POSE_NAME = 'mouse_approach'
CAPTURE_POSE = False
if CAPTURE_POSE:
    observation = client.snapshot()
    library.teach(POSE_NAME, observation)
    library.save()
    print('Saved:', POSE_NAME)
    client.show(observation)
else:
    print('Position the arm in the workbench, choose POSE_NAME, then set CAPTURE_POSE = True.')
print('Poses still needed:', library.missing())


## 2. Ask Codex for a plan

Set `CALL_CODEX = True` for one bounded cloud planning call. This sends the current
camera images and taught-skill description to Codex, but does not claim robot control.
Keep text instructions specific, for example "move the mouse onto the right gray area".


In [ ]:
TASK = 'Move the stuffed toy mouse onto the left gray area.'
CALL_CODEX = False
plan = None
if CALL_CODEX:
    if library.missing():
        print('Some poses are not yet taught:', library.missing())
        print('Codex may describe the scene and return needs_teaching without any executable skill.')
    observation = client.snapshot()
    client.show(observation)
    plan = planner.plan(observation, library, TASK)
    print({'status': plan.status, 'skills': plan.skills, 'reason': plan.reason,
           'elapsed_s': round(plan.elapsed_s, 3)})
else:
    print('Set CALL_CODEX = True for a read-only visual planning request. Teaching is required for motion.')


## 3. Review the proposed poses and duration

This cell constructs a trajectory without moving. It interpolates from the current
measured pose and uses a conservative speed. The initial gripper must already match
the first approach pose: set it while stationary through the workbench, then take a
new observation and plan. The helper refuses to combine a starting gripper change
with transport. Confirm that the robot, mouse and target
areas still match the plan, and inspect every transition in the physical workspace.


In [ ]:
trajectory = []
episode_seconds = 0.0
if plan is not None and plan.status == 'ready' and plan.skills:
    observation = client.snapshot()
    client.show(observation)
    print('Selected taught waypoints:', library.preview(plan.skills))
    trajectory = library.trajectory(plan.skills, start=observation.joints, hz=20, speed=8)
    episode_seconds = len(trajectory) / 20 + 1
    print({'action_rows': len(trajectory), 'episode_seconds': round(episode_seconds, 2)})
    if episode_seconds > 60:
        raise RuntimeError('The complete path exceeds 60 seconds. Execute a reviewed individual skill instead.')
else:
    print('No executable plan is selected.')


## 4. Explicitly execute during a supervised session

The updated workbench model API must be deployed. Release any browser control lease,
have an operator beside the arm, and keep STOP available. Physical testing was deferred
until that session. Run this cell only after reviewing the plan and path above.

Changing the mouse position after teaching invalidates the pickup path. If anything
changed since the preview, obtain another observation and plan first.


In [ ]:
EXECUTE = False
ARM_CONFIRMATION = ''  # Type ARM ROBOT only for the supervised run.
if EXECUTE:
    if plan is None or plan.status != 'ready' or not trajectory:
        raise RuntimeError('Create and inspect a ready plan and trajectory first.')
    records = client.run_targets(
        trajectory, hz=20, speed=8, seconds=episode_seconds,
        confirmation=ARM_CONFIRMATION,
    )
    print('Completed target updates:', len(records))
    client.show(client.snapshot())
else:
    print('Motion disabled. Set EXECUTE and the explicit arming phrase only when supervised.')


## Explicit STOP

The workbench's STOP button is always available. This cell makes the same hold request.
It revokes the current controller and keeps existing torque enabled.


In [ ]:
REQUEST_STOP = False
if REQUEST_STOP:
    client.stop()
    print('STOP requested.')
